# Camada Gold — Esquema Estrela
**Fonte:** ecommerce_mvp.silver.vendas_tratadas (397.884 linhas, já limpa)

**Fato:** `fato_vendas` — grão: item vendido (uma linha por produto dentro de um pedido)

**Dimensões:** `dim_produto`, `dim_cliente`, `dim_regiao`, `dim_tempo`

**Tabela derivada:** `agg_pedidos` — grão: pedido (usada para ticket médio e contagem de pedidos)

In [0]:
df_silver = spark.table("ecommerce_mvp.silver.vendas_tratadas")
display(df_silver.limit(5))                                    #Leitura da camada silver

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850,United Kingdom,15.299999999999999
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,20.34
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.65,17850,United Kingdom,15.3
536368,22913,RED COAT RACK PARIS FASHION,3,2010-12-01T08:34:00.000Z,4.95,13047,United Kingdom,14.850000000000001
536370,21791,VINTAGE HEADS AND TAILS CARD GAME,24,2010-12-01T08:45:00.000Z,1.25,12583,France,30.0


In [0]:
dim_produto = df_silver.select("StockCode", "Description").distinct()
dim_cliente = df_silver.select("CustomerID").distinct()
dim_regiao = df_silver.select("Country").distinct()
dim_tempo = (
    df_silver.select("InvoiceDate")
    .distinct()
    .selectExpr(
        "InvoiceDate",
        "year(InvoiceDate) as Ano",
        "month(InvoiceDate) as Mes",
        "dayofmonth(InvoiceDate) as Dia"
    )
)

dim_produto.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_produto")
dim_cliente.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_cliente")
dim_regiao.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_regiao")
dim_tempo.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_tempo")

print("Produtos distintos:", dim_produto.count())
print("Clientes distintos:", dim_cliente.count())
print("Países distintos:", dim_regiao.count())
print("Datas/horários distintos:", dim_tempo.count())

Produtos distintos: 3897
Clientes distintos: 4338
Países distintos: 37
Datas/horários distintos: 17282


In [0]:
stockcodes_unicos = df_silver.select("StockCode").distinct().count()
print("StockCodes únicos:", stockcodes_unicos)
print("Linhas em dim_produto:", dim_produto.count())

StockCodes únicos: 3665
Linhas em dim_produto: 3897


In [0]:
# Dimensão de produto: agrupa por StockCode + Description e conta ocorrências,
# pois um mesmo StockCode pode ter mais de uma Description no histórico (inconsistência textual)
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

# Ordena as descrições de cada StockCode pela mais frequente primeiro
window_desc = Window.partitionBy("StockCode").orderBy(col("count").desc())

dim_produto = (
    df_silver.groupBy("StockCode", "Description").count()
    .withColumn("rn", row_number().over(window_desc))
    .filter("rn = 1")  # mantém apenas a descrição mais frequente por StockCode
    .select("StockCode", "Description")
)
dim_produto.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_produto")

# Dimensão de cliente: um cliente por linha
dim_cliente = df_silver.select("CustomerID").distinct()
dim_cliente.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_cliente")

# Dimensão de região: um país por linha
dim_regiao = df_silver.select("Country").distinct()
dim_regiao.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_regiao")

# Dimensão de tempo: quebra InvoiceDate em ano, mês e dia para facilitar análises temporais
dim_tempo = (
    df_silver.select("InvoiceDate")
    .distinct()
    .selectExpr(
        "InvoiceDate",
        "year(InvoiceDate) as Ano",
        "month(InvoiceDate) as Mes",
        "dayofmonth(InvoiceDate) as Dia"
    )
)
dim_tempo.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.dim_tempo")

print("dim_produto (corrigida):", dim_produto.count())
print("dim_cliente:", dim_cliente.count())
print("dim_regiao:", dim_regiao.count())
print("dim_tempo:", dim_tempo.count())

dim_produto (corrigida): 3665
dim_cliente: 4338
dim_regiao: 37
dim_tempo: 17282


In [0]:
# Tabela fato: grão de item vendido (uma linha por produto dentro de um pedido)
# Mantém as chaves para as dimensões (StockCode, CustomerID, Country, InvoiceDate)
# e as métricas de negócio (Quantity, UnitPrice, TotalPrice)
fato_vendas = df_silver.select(
    "InvoiceNo", "StockCode", "CustomerID", "Country",
    "InvoiceDate", "Quantity", "UnitPrice", "TotalPrice"
)
fato_vendas.write.mode("overwrite").saveAsTable("ecommerce_mvp.gold.fato_vendas")

print("Linhas em fato_vendas:", fato_vendas.count())

Linhas em fato_vendas: 392692


In [0]:
# Tabela derivada: agrega o fato por pedido (InvoiceNo), somando o valor de todos os itens
# Grão diferente do fato: aqui cada linha é um pedido completo, não um item
from pyspark.sql.functions import sum as _sum, col

agg_pedidos = (
    fato_vendas
    .groupBy("InvoiceNo", "CustomerID", "Country", "InvoiceDate")
    .agg(_sum("TotalPrice").alias("ValorTotalPedido"))
)

# Ajuste de tipo: double -> decimal, evitando erro de arredondamento binário
agg_pedidos = agg_pedidos.withColumn(
    "ValorTotalPedido",
    col("ValorTotalPedido").cast("decimal(10,2)")
)

agg_pedidos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.agg_pedidos")
display(agg_pedidos.limit(10))

agg_pedidos.selectExpr(
    "avg(ValorTotalPedido) as ticket_medio",
    "count(*) as total_pedidos"
).show()

InvoiceNo,CustomerID,Country,InvoiceDate,ValorTotalPedido
536365,17850,United Kingdom,2010-12-01T08:26:00.000Z,139.12
536368,13047,United Kingdom,2010-12-01T08:34:00.000Z,70.05
536370,12583,France,2010-12-01T08:45:00.000Z,855.86
536373,17850,United Kingdom,2010-12-01T09:02:00.000Z,259.86
536375,17850,United Kingdom,2010-12-01T09:32:00.000Z,259.86
536378,14688,United Kingdom,2010-12-01T09:37:00.000Z,444.98
536381,15311,United Kingdom,2010-12-01T09:41:00.000Z,449.98
536384,18074,United Kingdom,2010-12-01T09:53:00.000Z,489.60
536387,16029,United Kingdom,2010-12-01T09:58:00.000Z,3193.92
536389,12431,Australia,2010-12-01T10:03:00.000Z,358.25


+------------+-------------+
|ticket_medio|total_pedidos|
+------------+-------------+
|  478.785093|        18562|
+------------+-------------+



In [0]:
# Agrupa o fato por produto e soma a quantidade vendida
# Junta com dim_produto para trazer a descrição legível (não só o código)
top_produtos = (
    fato_vendas.groupBy("StockCode")
    .agg(
        _sum("Quantity").alias("QuantidadeTotal"),
        _sum("TotalPrice").alias("FaturamentoTotal")
    )
    .withColumn("FaturamentoTotal", col("FaturamentoTotal").cast("decimal(12,2)"))
    .join(dim_produto, on="StockCode", how="left")
    .select("StockCode", "Description", "QuantidadeTotal", "FaturamentoTotal")
    .orderBy(col("QuantidadeTotal").desc())
)
top_produtos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.top_produtos")
display(top_produtos.limit(10))

StockCode,Description,QuantidadeTotal,FaturamentoTotal
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54319,13558.41
22197,POPCORN HOLDER,49160,37206.88
85099B,JUMBO BAG RED RETROSPOT,46078,85040.54
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36763,100547.45
84879,ASSORTED COLOUR BIRD ORNAMENT,35263,56413.03
21212,PACK OF 72 RETROSPOT CAKE CASES,33670,16381.88
23084,RABBIT NIGHT LIGHT,27153,51251.24
22492,MINI PAINT SET VINTAGE,26076,16039.24


## Síntese — Camada Gold (Esquema Estrela)

**Fonte:** `ecommerce_mvp.silver.vendas_tratadas` (397.884 linhas)

### Estrutura criada
- **Fato** `fato_vendas` — grão de item vendido, 397.884 linhas (herdadas diretamente da silver, sem filtro adicional)
- **Dimensões:**
  - `dim_produto` — 3.665 produtos únicos
  - `dim_cliente` — 4.338 clientes únicos
  - `dim_regiao` — 37 países
  - `dim_tempo` — 17.282 combinações de data/hora distintas
- **Tabela derivada** `agg_pedidos` — grão de pedido (agregação de `fato_vendas` por `InvoiceNo`), 18.562 pedidos

### Problema de qualidade identificado e resolvido
Ao construir `dim_produto`, foram encontrados 232 `StockCode` associados a mais de uma `Description` distinta (inconsistência textual, provavelmente por atualização de nome do produto ao longo do tempo). Resolvido mantendo, para cada `StockCode`, a `Description` mais frequente no histórico — reduzindo a dimensão de 3.897 para 3.665 linhas (uma por produto, como esperado).

### Ajuste técnico de tipos monetários
Colunas de valor (`ValorTotalPedido`, `FaturamentoTotal`) foram convertidas de `double` para `decimal(10,2)`/`decimal(12,2)`, evitando erros de arredondamento binário (ex: valores como `22.200000000000003`) em métricas financeiras.

### Métricas gerais observadas
- Ticket médio por pedido: **£480,09**
- Total de pedidos no período: **18.562**

### Tabelas de resposta às perguntas do MVP
- `top_produtos` — produtos mais vendidos em quantidade
- `vendas_por_regiao` — volume e faturamento por país
- `top_clientes` — maiores clientes por faturamento

# Qualidade de Dados e Análise Final
## Parte 1: Qualidade de Dados (dados brutos, camada bronze)
Fonte: `ecommerce_mvp.bronze.data`, sem nenhuma transformação aplicada — avaliação do "processo inicial de captura".

In [0]:
# Lê os dados exatamente como capturados, sem qualquer tratamento
df_bronze = spark.table("ecommerce_mvp.bronze.data")
total_linhas = df_bronze.count()
print(f"Total de linhas (dados brutos): {total_linhas}")

Total de linhas (dados brutos): 541909


## Completude

In [0]:
from pyspark.sql.functions import col, count, when

colunas = df_bronze.columns
tipos = dict(df_bronze.dtypes)  # mapa coluna -> tipo (ex: 'string', 'bigint', 'double')

expressoes = []
for c in colunas:
    if tipos[c] == "string":
        expressoes.append(count(when(col(c).isNull() | (col(c) == ""), c)).alias(c))
    else:
        expressoes.append(count(when(col(c).isNull(), c)).alias(c))

completude = df_bronze.select(expressoes)
display(completude)

for c in colunas:
    qtd_nulos = completude.select(c).collect()[0][0]
    proporcao = (qtd_nulos / total_linhas) * 100
    print(f"{c}: {qtd_nulos} nulos/vazios ({proporcao:.2f}%)")

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,0,1454,0,0,0,135080,0


InvoiceNo: 0 nulos/vazios (0.00%)
StockCode: 0 nulos/vazios (0.00%)
Description: 1454 nulos/vazios (0.27%)
Quantity: 0 nulos/vazios (0.00%)
InvoiceDate: 0 nulos/vazios (0.00%)
UnitPrice: 0 nulos/vazios (0.00%)
CustomerID: 135080 nulos/vazios (24.93%)
Country: 0 nulos/vazios (0.00%)


## Consistência

In [0]:
from pyspark.sql.functions import to_timestamp

# Tenta converter InvoiceDate no formato esperado; se falhar, vira nulo
# Isso revela quantos registros NÃO seguem o padrão "M/d/yyyy H:mm"
df_teste_data = df_bronze.withColumn(
    "data_convertida", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

inconsistentes = df_teste_data.filter(col("data_convertida").isNull()).count()
print(f"Registros com InvoiceDate fora do padrão esperado: {inconsistentes} ({(inconsistentes/total_linhas)*100:.2f}%)")

Registros com InvoiceDate fora do padrão esperado: 0 (0.00%)


In [0]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

# Padrão esperado: 5 dígitos, opcionalmente seguidos por 1 letra (ex: 85123A, 71053)
def valida_stockcode(codigo):
    if codigo is None:
        return False
    return bool(re.fullmatch(r"\d{5}[A-Za-z]?", codigo))

valida_stockcode_udf = udf(valida_stockcode, BooleanType())

df_teste_stockcode = df_bronze.withColumn("stockcode_valido", valida_stockcode_udf(col("StockCode")))

fora_padrao = df_teste_stockcode.filter(col("stockcode_valido") == False).count()
print(f"StockCodes fora do padrão esperado (5 dígitos + letra opcional): {fora_padrao} ({(fora_padrao/total_linhas)*100:.2f}%)")

# Mostra alguns exemplos dos que fogem do padrão, para entender o motivo
display(df_teste_stockcode.filter(col("stockcode_valido") == False).select("StockCode").distinct().limit(20))

StockCodes fora do padrão esperado (5 dígitos + letra opcional): 3385 (0.62%)


StockCode
POST
DCGS0070
gift_0001_50
BANK CHARGES
DCGS0055
AMAZONFEE
15056BL
D
C2
gift_0001_20


In [0]:
display(
    df_teste_stockcode
    .filter(col("stockcode_valido") == False)
    .groupBy("StockCode")
    .count()
    .orderBy(col("count").desc())
)

StockCode,count
POST,1256
DOT,710
M,571
15056BL,326
C2,144
D,77
S,63
15056bl,62
BANK CHARGES,37
AMAZONFEE,34


## Medindo impacto de linhas que não são produtos na camada gold

In [0]:
from pyspark.sql.functions import sum as _sum, count

# Lista de códigos confirmados como não-produto (ajustes administrativos/financeiros)
codigos_nao_produto = ["POST", "DOT", "M", "m", "C2", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"]

# Filtra o fato apenas com esses códigos, para medir o quanto eles representam
impacto = fato_vendas.filter(col("StockCode").isin(codigos_nao_produto))

qtd_linhas = impacto.count()
qtd_total_vendida = impacto.agg(_sum("Quantity")).collect()[0][0]
faturamento_total = impacto.agg(_sum("TotalPrice")).collect()[0][0]

print(f"Linhas no fato com códigos não-produto: {qtd_linhas} ({(qtd_linhas/fato_vendas.count())*100:.2f}% do fato)")
print(f"Quantidade total (não-produto): {qtd_total_vendida}")
print(f"Faturamento total (não-produto): £{faturamento_total:.2f}")

# Para efeito de comparação, o faturamento total do fato inteiro
faturamento_geral = fato_vendas.agg(_sum("TotalPrice")).collect()[0][0]
print(f"Faturamento total do fato (todos os produtos): £{faturamento_geral:.2f}")
print(f"Participação dos códigos não-produto no faturamento: {(faturamento_total/faturamento_geral)*100:.2f}%")

Linhas no fato com códigos não-produto: 1539 (0.39% do fato)
Quantidade total (não-produto): 10215
Faturamento total (não-produto): £149981.25
Faturamento total do fato (todos os produtos): £8887208.89
Participação dos códigos não-produto no faturamento: 1.69%


## Unicidade

In [0]:
# Verifica se existem linhas 100% duplicadas no dado bruto
duplicatas_totais = df_bronze.count() - df_bronze.distinct().count()
print(f"Linhas totalmente duplicadas: {duplicatas_totais} ({(duplicatas_totais/total_linhas)*100:.2f}%)")

# Verifica duplicidade mais específica: mesmo produto, no mesmo pedido, aparecendo mais de uma vez
# (o que teoricamente não deveria acontecer - um item deveria aparecer só uma vez por nota fiscal)
duplicatas_item_pedido = (
    df_bronze.groupBy("InvoiceNo", "StockCode")
    .count()
    .filter(col("count") > 1)
)
qtd_casos = duplicatas_item_pedido.count()
print(f"Combinações InvoiceNo+StockCode repetidas: {qtd_casos}")
display(duplicatas_item_pedido.orderBy(col("count").desc()).limit(10))

Linhas totalmente duplicadas: 5268 (0.97%)
Combinações InvoiceNo+StockCode repetidas: 9694


InvoiceNo,StockCode,count
555524,22698,20
C544580,S,16
555524,22697,12
572861,22775,8
572344,M,7
C544583,S,7
C553531,S,7
578289,23395,7
541266,21755,6
540524,21756,6


In [0]:
# Pega um caso de combinação repetida e mostra todas as linhas dele, para ver se os valores diferem
exemplo = duplicatas_item_pedido.orderBy(col("count").desc()).first()
print(f"Exemplo: InvoiceNo={exemplo['InvoiceNo']}, StockCode={exemplo['StockCode']}, ocorrências={exemplo['count']}")

df_bronze.filter(
    (col("InvoiceNo") == exemplo["InvoiceNo"]) & (col("StockCode") == exemplo["StockCode"])
).show(truncate=False)

Exemplo: InvoiceNo=555524, StockCode=22698, ocorrências=20
+---------+---------+------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+------------------------------+--------+--------------+---------+----------+--------------+
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |6/5/2011 11:37|2.95     |16923     |United Kingdom|
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |6/5/2011 11:37|2.95     |16923     |United Kingdom|
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |6/5/2011 11:37|2.95     |16923     |United Kingdom|
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |6/5/2011 11:37|2.95     |16923     |United Kingdom|
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |6/5/2011 11:37|2.95     |16923     |United Kingdom|
|555524   |22698    |PINK REGENCY TEA

In [0]:
# Verifica quantas dessas combinações repetidas têm TODOS os outros campos idênticos também
# (ou seja, são duplicatas completas, não variações legítimas)
duplicatas_completas_dentro_combinacao = (
    df_bronze.groupBy("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate", "CustomerID")
    .count()
    .filter(col("count") > 1)
)

linhas_afetadas = duplicatas_completas_dentro_combinacao.selectExpr("sum(count) as total").collect()[0]["total"]
print(f"Linhas envolvidas em duplicação completa (todos os campos iguais): {linhas_afetadas}")

Linhas envolvidas em duplicação completa (todos os campos iguais): 10151


## Acurácia

In [0]:
# Verifica se existem valores fisicamente implausíveis
# Quantity ou UnitPrice extremos podem indicar erro de digitação (ex: preço de £10000 para um item de £1)
df_bronze.select("Quantity", "UnitPrice").describe().show()

# Casos extremos específicos, para inspeção manual
print("Maiores quantidades:")
df_bronze.orderBy(col("Quantity").desc()).select("StockCode", "Description", "Quantity", "UnitPrice").show(5, truncate=False)

print("Maiores preços unitários:")
df_bronze.orderBy(col("UnitPrice").desc()).select("StockCode", "Description", "Quantity", "UnitPrice").show(5, truncate=False)

+-------+------------------+-----------------+
|summary|          Quantity|        UnitPrice|
+-------+------------------+-----------------+
|  count|            541909|           541909|
|   mean|  9.55224954743324|4.611113626083471|
| stddev|218.08115785023284|96.75985306117848|
|    min|            -80995|        -11062.06|
|    max|             80995|          38970.0|
+-------+------------------+-----------------+

Maiores quantidades:
+---------+---------------------------------+--------+---------+
|StockCode|Description                      |Quantity|UnitPrice|
+---------+---------------------------------+--------+---------+
|23843    |PAPER CRAFT , LITTLE BIRDIE      |80995   |2.08     |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR   |74215   |1.04     |
|84826    |ASSTD DESIGN 3D PAPER STICKERS   |12540   |0.0      |
|37413    |NULL                             |5568    |0.0      |
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS|4800    |0.21     |
+---------+------------------------

In [0]:
# Verifica se existe um cancelamento espelhado para a venda de 80.995 unidades
df_bronze.filter(col("StockCode") == "23843").orderBy(col("Quantity").desc()).select(
    "InvoiceNo", "StockCode", "Quantity", "UnitPrice", "InvoiceDate", "CustomerID"
).show(10, truncate=False)

+---------+---------+--------+---------+--------------+----------+
|InvoiceNo|StockCode|Quantity|UnitPrice|InvoiceDate   |CustomerID|
+---------+---------+--------+---------+--------------+----------+
|581483   |23843    |80995   |2.08     |12/9/2011 9:15|16446     |
|C581484  |23843    |-80995  |2.08     |12/9/2011 9:27|16446     |
+---------+---------+--------+---------+--------------+----------+



## Outliers

In [0]:
from pyspark.sql.functions import mean, stddev, abs as _abs

# Calcula média e desvio padrão de Quantity e UnitPrice (usando só valores positivos, já que negativos são outro problema já tratado)
stats = df_bronze.filter((col("Quantity") > 0) & (col("UnitPrice") > 0)).select(
    mean("Quantity").alias("media_qtd"), stddev("Quantity").alias("std_qtd"),
    mean("UnitPrice").alias("media_preco"), stddev("UnitPrice").alias("std_preco")
).collect()[0]

# Define outlier como valor a mais de 3 desvios-padrão da média (regra prática comum)
limite_qtd = stats["media_qtd"] + 3 * stats["std_qtd"]
limite_preco = stats["media_preco"] + 3 * stats["std_preco"]

print(f"Limite superior Quantity (média + 3 desvios): {limite_qtd:.2f}")
print(f"Limite superior UnitPrice (média + 3 desvios): {limite_preco:.2f}")

outliers_qtd = df_bronze.filter(col("Quantity") > limite_qtd).count()
outliers_preco = df_bronze.filter(col("UnitPrice") > limite_preco).count()

print(f"Registros com Quantity acima do limite: {outliers_qtd} ({(outliers_qtd/total_linhas)*100:.2f}%)")
print(f"Registros com UnitPrice acima do limite: {outliers_preco} ({(outliers_preco/total_linhas)*100:.2f}%)")

Limite superior Quantity (média + 3 desvios): 477.11
Limite superior UnitPrice (média + 3 desvios): 111.65
Registros com Quantity acima do limite: 542 (0.10%)
Registros com UnitPrice acima do limite: 1001 (0.18%)


In [0]:
codigos_nao_produto = ["POST", "DOT", "M", "m", "C2", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"]

outliers_preco_admin = df_bronze.filter(
    (col("UnitPrice") > 111.65) & (col("StockCode").isin(codigos_nao_produto))
).count()

print(f"Outliers de preço que são códigos administrativos: {outliers_preco_admin} de 1001 ({(outliers_preco_admin/1001)*100:.2f}%)")

Outliers de preço que são códigos administrativos: 875 de 1001 (87.41%)


%md
## Síntese — Qualidade de Dados (dados brutos, camada bronze)
Fonte: `ecommerce_mvp.bronze.data`, 541.909 linhas, sem nenhuma transformação.

### Completude
- `CustomerID`: 135.080 nulos (24,93%) — vendas sem cliente identificado
- `Description`: 1.454 nulos (0,27%)
- Demais colunas: 0% de nulos/vazios

### Consistência
- `InvoiceDate`: 100% dos registros seguem o padrão `M/d/yyyy H:mm`
- `StockCode`: 3.385 registros (0,62%) fora do padrão numérico esperado — investigação revelou que a maioria são códigos administrativos legítimos (POST, DOT, M, BANK CHARGES, AMAZONFEE, entre outros), não produtos físicos

### Unicidade
- 5.268 linhas totalmente duplicadas (0,97%) — mesmo InvoiceNo, StockCode, Quantity, UnitPrice, InvoiceDate e CustomerID repetidos, sem explicação de negócio válida (ex: mesma xícara "comprada" 20 vezes no mesmo minuto)
- **Tratamento:** `dropDuplicates()` adicionado à camada silver, reduzindo o total de 397.884 para 392.692 linhas

### Acurácia
- `UnitPrice` negativo (mínimo -£11.062,06) — tratado pelo filtro `UnitPrice > 0` na silver
- Maiores preços unitários concentrados em códigos administrativos (ex: "M"/Manual a £38.970, AMAZONFEE)
- Caso identificado: venda de 80.995 unidades do produto 23843 (pedido 581483), cancelada 12 minutos depois (pedido C581484) — permanece na silver por limitação do filtro atual (ver Trabalhos Futuros)

### Outliers
- 0,10% dos registros com `Quantity` acima de 3 desvios-padrão da média
- 0,18% dos registros com `UnitPrice` acima de 3 desvios-padrão da média — 87,41% desses são explicados pelos mesmos códigos administrativos identificados na Consistência

### Conclusão geral
Os principais achados de qualidade (StockCodes administrativos e duplicação de linhas) têm causa identificável e tratamento aplicado ou justificado. O impacto financeiro dos códigos administrativos na camada gold foi medido (1,69% do faturamento total) e será tratado especificamente na tabela `top_produtos`, mantido em `vendas_por_regiao` e `top_clientes` (ver justificativa na seção de Análise Final).

## Pergunta 1: Quais são os produtos mais vendidos em quantidade e por faturamento?

In [0]:
from pyspark.sql.functions import countDistinct

# Agrupa o fato por produto, excluindo:
# - códigos administrativos (não representam vendas de produto real)
# - o pedido cancelado 581483 (mesma correção aplicada nas demais tabelas gold)
top_produtos = (
    fato_vendas
    .filter(~col("StockCode").isin(codigos_nao_produto))
    .filter(~col("InvoiceNo").isin(["581483"]))
    .groupBy("StockCode")
    .agg(
        _sum("Quantity").alias("QuantidadeTotal"),           # soma de itens vendidos por produto
        _sum("TotalPrice").alias("FaturamentoTotal"),         # soma do valor gerado por produto
        countDistinct("InvoiceNo").alias("TotalPedidos")      # em quantos pedidos distintos o produto aparece
    )
    # Ajuste de tipo: double -> decimal, evitando erro de arredondamento binário
    .withColumn("FaturamentoTotal", col("FaturamentoTotal").cast("decimal(12,2)"))
    # Ticket médio: valor médio do pedido completo, entre os pedidos que contêm este produto
    # (não é o preço do produto isolado, é o valor total do pedido em que ele aparece)
    .withColumn("TicketMedio", (col("FaturamentoTotal") / col("TotalPedidos")).cast("decimal(10,2)"))
    # Junta com dim_produto para trazer a descrição legível do produto (não só o código)
    .join(dim_produto, on="StockCode", how="left")
    .select("StockCode", "Description", "QuantidadeTotal", "FaturamentoTotal", "TotalPedidos", "TicketMedio")
    # Ordena pela quantidade, critério da pergunta 1 do MVP
    .orderBy(col("QuantidadeTotal").desc())
)
top_produtos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.top_produtos")
display(top_produtos.limit(10))

StockCode,Description,QuantidadeTotal,FaturamentoTotal,TotalPedidos,TicketMedio
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,195,417.52
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54319,13558.41,472,28.73
22197,POPCORN HOLDER,49160,37206.88,1035,35.95
85099B,JUMBO BAG RED RETROSPOT,46078,85040.54,1600,53.15
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36763,100547.45,1978,50.83
84879,ASSORTED COLOUR BIRD ORNAMENT,35263,56413.03,1375,41.03
21212,PACK OF 72 RETROSPOT CAKE CASES,33670,16381.88,1029,15.92
23084,RABBIT NIGHT LIGHT,27153,51251.24,801,63.98
22492,MINI PAINT SET VINTAGE,26076,16039.24,325,49.35
22616,PACK OF 12 LONDON TISSUES,25329,7261.77,382,19.01


## Pergunta 2: Quais regiões concentram as vendas em volume e por faturamento?

In [0]:
from pyspark.sql.functions import avg, count

# Exclui o pedido cancelado (mesma correção aplicada em top_produtos e top_clientes)
agg_pedidos = agg_pedidos.filter(col("InvoiceNo") != "581483")
agg_pedidos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.agg_pedidos")

# Recalcula ticket médio e total de pedidos por região com o dado corrigido
ticket_por_regiao = (
    agg_pedidos.groupBy("Country")
    .agg(
        _sum("ValorTotalPedido").alias("FaturamentoTotal"),
        count("InvoiceNo").alias("TotalPedidos"),
        avg("ValorTotalPedido").alias("TicketMedio")
    )
    .withColumn("FaturamentoTotal", col("FaturamentoTotal").cast("decimal(12,2)"))
    .withColumn("TicketMedio", col("TicketMedio").cast("decimal(10,2)"))
)

# Recria vendas_por_regiao a partir do fato (também sem o pedido cancelado) + ticket médio corrigido
vendas_por_regiao = (
    fato_vendas
    .filter(col("InvoiceNo") != "581483")
    .groupBy("Country")
    .agg(
        _sum("Quantity").alias("VolumeTotal"),
        _sum("TotalPrice").alias("FaturamentoTotal")
    )
    .withColumn("FaturamentoTotal", col("FaturamentoTotal").cast("decimal(12,2)"))
    .join(ticket_por_regiao.select("Country", "TotalPedidos", "TicketMedio"), on="Country", how="left")
    .orderBy(col("FaturamentoTotal").desc())
)

vendas_por_regiao.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.vendas_por_regiao")
display(vendas_por_regiao.limit(10))

Country,VolumeTotal,FaturamentoTotal,TotalPedidos,TicketMedio
United Kingdom,4160310,7116555.04,16672,426.86
Netherlands,200361,285446.34,94,3036.66
EIRE,140133,265262.46,261,1016.33
Germany,119154,228678.40,457,500.39
France,111428,208934.31,390,535.73
Australia,83891,138453.81,57,2429.01
Spain,27933,61558.56,90,683.98
Switzerland,30082,56443.95,51,1106.74
Belgium,23237,41196.34,98,420.37
Sweden,36078,38367.83,36,1065.77


## Pergunta 3: Quais são os maiores clientes em termos de faturamento?

In [0]:
from pyspark.sql.functions import first

# Agrupa agg_pedidos (grão de pedido) por cliente, para calcular ticket médio real por cliente
# agg_pedidos já está sem o pedido cancelado 581483 (correção aplicada anteriormente)
ticket_por_cliente = (
    agg_pedidos.groupBy("CustomerID")
    .agg(
        count("InvoiceNo").alias("TotalPedidos"),      # quantos pedidos distintos o cliente fez
        avg("ValorTotalPedido").alias("TicketMedio")    # valor médio dos pedidos do cliente
    )
    .withColumn("TicketMedio", col("TicketMedio").cast("decimal(10,2)"))
)

# Agrupa o fato por cliente, somando faturamento e trazendo o país associado
top_clientes = (
    fato_vendas
    .filter(~col("InvoiceNo").isin(["581483"]))  # mesma correção do pedido cancelado
    .groupBy("CustomerID")
    .agg(
        _sum("TotalPrice").alias("FaturamentoTotal"),
        first("Country").alias("Country")
    )
    .withColumn("FaturamentoTotal", col("FaturamentoTotal").cast("decimal(12,2)"))
    # Junta com o ticket médio calculado a partir de agg_pedidos
    .join(ticket_por_cliente, on="CustomerID", how="left")
    .select("CustomerID", "Country", "FaturamentoTotal", "TotalPedidos", "TicketMedio")
    .orderBy(col("FaturamentoTotal").desc())
)
top_clientes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_mvp.gold.top_clientes")
display(top_clientes.limit(10))

CustomerID,Country,FaturamentoTotal,TotalPedidos,TicketMedio
14646,Netherlands,280206.02,73,3838.44
18102,United Kingdom,259657.30,60,4327.62
17450,United Kingdom,194390.79,46,4225.89
14911,EIRE,143711.17,202,711.44
12415,Australia,124914.53,21,5948.31
14156,EIRE,117210.08,55,2131.09
17511,United Kingdom,91062.38,32,2845.70
16029,United Kingdom,80850.84,63,1283.35
12346,United Kingdom,77183.60,1,77183.60
16684,United Kingdom,66653.56,28,2380.48
